# 01 · Explore the raw data

**Goal:** find every kind of dirty data in the raw payment files *before* writing the pipeline.

**How this notebook is organised**
- Each hunt in Step 5 ends with a notes cell that answers what is wrong, whether it can be fixed with certainty, and which one-line check catches it.
- `data_small/answer_key/` is only opened in Step 7, after I found the defects without it.

**The layers**
```
raw/  (dirty, as received)  ──►  pipeline  ──►  clean/  (trusted)

```

## Step 1 · Setup
Imports and paths. `ROOT` is the project folder; everything else is relative to it, so the notebook works no matter where Jupyter was started.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data_small" / "raw"
CSV = ROOT / "data_small" / "exports" / "csv"            # dimension + fact tables as CSV files
CLEAN_DB = ROOT / "data_small" / "clean" / "treasury.sqlite"
ANSWER_KEY = ROOT / "data_small" / "answer_key"          # only opened in Step 7

con = sqlite3.connect(CLEAN_DB)
print(ROOT)
print(sorted(p.name for p in RAW.iterdir()))

<project root>
['bank_statements', 'invoices', 'payment_events', 'payments']


## Step 2 · Load the raw payment files

Each feed is a folder with **one Parquet file per month**. Parquet is a compressed, typed column format: unlike CSV, it remembers that `amount` is a number and `initiated_ts` is a timestamp.

**2a:** read one month to see the shape.

In [2]:
one = pd.read_parquet(RAW / "payments" / "payments_2024-10.parquet")
print(one.shape)
one.head()

(4204, 30)


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff
0,P00008586,E2E00008586,0688db66-15fd-45db-87da-dd7d3b27f5a4,NaN,OUT,A001,C0075,11,API,SG,XB,12007.92,USD,16141.12,1.347567,SHA,18.60,11994.32,SUPP,C0075/330970,False,2024-10-01 01:05:00,2024-10-01 01:35:04,2024-10-01 01:42:28,20241001,completed,NaN,True,0,False
1,P00008587,E2E00008587,621b48c9-12b4-4193-b24b-88b9161bb94d,NaN,OUT,A004,C0120,11,API,SG,GB,10739.70,GBP,18108.73,1.690364,SHA,14.83,10711.38,SUPP,C0120/486024,False,2024-10-01 01:09:00,2024-10-01 01:27:09,2024-10-01 02:29:00,20241001,completed,NaN,True,0,False
2,P00008588,E2E00008588,NaN,E02-2-20241001,OUT,A007,C0186,2,API,SG,SG,16055.54,SGD,16055.54,NaN,NaN,0.20,16055.54,SUPP,C0186/582672,False,2024-10-01 01:33:00,2024-10-01 01:44:57,2024-10-02 01:44:00,20241002,completed,NaN,True,0,False
3,P00008589,E2E00008589,67cd9fb7-ab62-48f1-9a49-dfe3febe4222,NaN,OUT,A004,C0137,11,API,SG,CN,338958.55,CNY,62650.39,0.185941,SHA,135.26,338876.35,SUPP,C0137/850475,False,2024-10-01 01:39:00,2024-10-01 01:45:02,2024-10-08 01:11:00,20241008,completed,NaN,True,0,False
4,P00008590,E2E00008590,5ffbd641-ec08-47b6-8250-a70e9076b410,NaN,OUT,A004,C0138,11,API,SG,CN,13108.55,CNY,2422.88,0.185941,SHA,135.26,12768.76,SUPP,C0138/84768,False,2024-10-01 01:50:00,2024-10-01 02:06:04,2024-10-08 01:45:00,20241008,completed,NaN,True,0,False


**2b:** read **all** months of `payments` into one DataFrame called `raw`.

Every month has its own parquet file (e.g. `.../data_small/raw/payments/payments_2025-03.parquet`). This step combines all of them into one table.

Steps:
1. List the files with `sorted((RAW / "payments").glob("payments_*.parquet"))`
2. Read each file with `pd.read_parquet` and add a column `_source_file` holding the file name (so I can say which file a bad row came from)
3. Stack them with `pd.concat(..., ignore_index=True)`

Result: `len(raw)` is **106,621** rows.

This becomes `extract.read_raw()` in the pipeline later.

In [3]:
files = sorted((RAW / "payments").glob("payments_*.parquet")) # Lists all the payment files
raw = pd.concat([pd.read_parquet(f).assign(_source_file=f.name) for f in files], ignore_index=True) # Reads each month, tags it with its file name, then stacks them all into "raw"

len(raw) # counts the number of rows

106621

## Step 3 · First look

Get a feel for the table before hunting problems:

1. What columns and types are there? → `raw.dtypes`
2. What does a typical row look like? → `raw.sample(5)`
3. Which columns have missing values, and how many? → `raw.isna().sum().sort_values(ascending=False)`
4. What are the ranges of the numbers? → `raw.describe()`

⚠️ **Missing isn't always wrong.** `uetr` is only set for cross-border payments, `settled_ts` is empty for payments still pending, and `failure_reason` only exists for failures. For each column with nulls, I decide: *expected* or *defect*?

In [4]:
raw.dtypes


payment_id                     str
end_to_end_id                  str
uetr                           str
batch_id                    object
direction                      str
account_id                     str
counterparty_id                str
type_id                      int64
channel                        str
sender_country                 str
receiver_country               str
amount                     float64
currency_code                  str
amount_sgd                 float64
fx_rate_applied             object
charge_bearer                  str
fees                       float64
amount_received            float64
purpose_code                   str
remittance_ref              object
is_intercompany               bool
initiated_ts        datetime64[ms]
submitted_ts        datetime64[us]
settled_ts          datetime64[ms]
value_date_id                Int64
status                         str
failure_reason              object
is_stp                        bool
repair_count        

In [5]:
raw.sample(5)

,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file
91627,P00097143,E2E00097143,NaN,NaN,IN,A021,C0235,4,INBOUND,DE,DE,5587.99,EUR,8078.50,NaN,NaN,0.0,5587.99,GDDS,INV-E06-0032476,False,2026-06-30 11:20:00,2026-06-30 11:24:19,2026-07-01 08:42:00,20260701,completed,NaN,True,0,False,payments_2026-06.parquet
102607,P00107624,E2E00107624,NaN,NaN,OUT,A034,C0084,12,PORTAL,GB,NL,70.44,GBP,119.90,NaN,NaN,0.0,70.44,GDDS,NaN,False,2026-09-07 09:37:00,2026-09-07 12:33:53,2026-09-09 10:54:00,20260909,completed,NaN,True,0,False,payments_2026-09.parquet
22811,P00030625,E2E00030625,NaN,NaN,IN,A032,C0058,5,INBOUND,NL,NL,16235.97,EUR,23504.31,NaN,NaN,0.0,16235.97,SCVE,INV-E08-0008610,False,2025-03-18 13:12:00,2025-03-18 13:15:29,2025-03-18 13:16:14,20250318,completed,NaN,True,0,False,payments_2025-03.parquet
82527,P00088320,E2E00088320,NaN,NaN,OUT,A020,C0005,12,H2H_FILE,DE,IN,40.71,EUR,59.94,NaN,NaN,0.0,40.71,GDDS,NaN,False,2026-04-30 13:18:00,2026-04-30 15:07:19,2026-05-05 09:12:00,20260505,completed,NaN,True,0,False,payments_2026-04.parquet
33658,P00041044,E2E00041044,NaN,NaN,OUT,A020,C0196,12,H2H_FILE,DE,DE,141.81,EUR,207.32,NaN,NaN,0.0,141.81,GDDS,NaN,False,2025-06-03 09:25:00,2025-06-03 10:07:44,2025-06-05 09:55:00,20250605,completed,NaN,True,0,False,payments_2025-06.parquet


In [6]:
raw.isna().sum().sort_values(ascending=False) # Checks each column for a null value, sums and sorts it

failure_reason      105306
batch_id             98434
fx_rate_applied      94607
charge_bearer        67396
uetr                 67396
remittance_ref       39283
purpose_code          3167
amount_received       1346
value_date_id         1346
settled_ts            1346
is_stp                   0
submitted_ts             0
initiated_ts             0
is_intercompany          0
repair_count             0
missed_cutoff            0
status                   0
payment_id               0
fees                     0
end_to_end_id            0
amount_sgd               0
currency_code            0
amount                   0
receiver_country         0
sender_country           0
channel                  0
type_id                  0
counterparty_id          0
account_id               0
direction                0
_source_file             0
dtype: int64

In [7]:
raw.describe()

,type_id,amount,amount_sgd,fees,amount_received,initiated_ts,submitted_ts,settled_ts,value_date_id,repair_count
count,106621.000000,1.066210e+05,1.066210e+05,106621.000000,1.052750e+05,106621,106621,105275,105275.0,106621.000000
mean,10.188621,1.444407e+05,4.482627e+04,15.963063,1.460654e+05,2025-10-11 18:20:12.773000,2025-10-11 19:27:41.215014,2025-10-12 08:05:24.436000,20253443.250715,0.034440
min,1.000000,-1.534287e+07,1.900000e+00,0.000000,1.660000e+00,2024-10-01 01:05:00,2024-10-01 01:27:09,2024-10-01 01:42:28,20241001.0,0.000000
25%,10.000000,6.876800e+02,3.733600e+02,0.000000,6.979800e+02,2025-04-14 11:17:00,2025-04-14 12:04:39,2025-04-15 03:03:45.500000,20250415.0,0.000000
50%,11.000000,1.082106e+04,9.332390e+03,0.000000,1.086838e+04,2025-10-14 20:44:00,2025-10-14 21:35:33,2025-10-15 09:27:00,20251015.0,0.000000
75%,12.000000,5.145734e+04,3.487742e+04,0.140000,5.169000e+04,2026-04-14 02:55:00,2026-04-14 04:22:46,2026-04-14 06:12:49,20260414.0,0.000000
max,13.000000,9.700581e+07,4.200000e+06,4583.120000,9.700581e+07,2026-10-01 01:58:00,2026-10-01 02:14:13,2026-10-01 03:59:00,20261001.0,1.000000
std,2.620872,1.031560e+06,1.330695e+05,94.348375,1.036211e+06,NaN,NaN,NaN,6384.526255,0.182357


From the "na" check, we can see 10 columns with prominent null values:

failure_reason [105306], 

batch_id [98434],

fx_rate_applied [94607],

charge_bearer [67396],

uetr [67396],

remittance_ref [39283],

purpose_code [3167],

amount_received [1346],

value_date_id [1346],

settled_ts [1346].

Signficant and needs investigation: 
1. `purpose_code` is always a necessary field to populate (in accordance with ISO 20022???)

Insignificant and is expected (these are hypotheses, each one is tested in the checks below):
1. `uetr` (Unique End-to-End Transaction Reference) is only set for cross-border payments (dim_payment_type is_cross_border = 1)
2. `charge_bearer` (who pays the bank fees: OUR / BEN / SHA) is also only set for cross-border payments. It has nothing to do with whether a fee exists
3. `failure_reason` only exists for failures (status = rejected or returned)
4. `fx_rate_applied` only exists when the payment currency differs from the paying account's currency, and the payment is not rejected or pending
5. `amount_received` is null if the payment was rejected or is still pending
6. `value_date_id` and `settled_ts` are null if the payment was rejected or is still pending
7. `remittance_ref` is a reference which tells the receiver additional information, so nulls are to be expected
8. `batch_id` says which bulk file a payment was submitted in. It is only set for outbound payments on batch rails (not card), so nulls are expected

Results of the checks are summarised in the table at the end of the null-rule checks.

# First, check for duplicate payment id's

In [8]:
raw.duplicated().sum()


np.int64(415)

In [9]:
raw[raw.duplicated(subset=['payment_id'])]

,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file
4188,P00008598,E2E00008598,29d85190-de9c-4846-a7fd-04f7804d28dd,NaN,IN,A028,C0108,11,INBOUND,ID,DE,7059.55,US$,9489.49,NaN,BEN,0.00,7030.63,GDDS,NaN,False,2024-10-01 03:02:00,2024-10-01 03:06:28,2024-10-01 03:44:58,20241001,completed,NaN,True,0,False,payments_2024-10.parquet
4189,P00008723,E2E00008723,NaN,NaN,OUT,A020,C0140,12,H2H_FILE,DE,DE,233.01,EUR,345.76,NaN,NaN,0.00,233.01,GDDS,NaN,False,2024-10-01 10:43:00,2024-10-01 16:07:49,2024-10-04 07:15:00,20241004,completed,NaN,False,1,False,payments_2024-10.parquet
4190,P00008829,E2E00008829,NaN,NaN,OUT,A020,C0134,12,H2H_FILE,DE,DE,177.51,EUR,264.12,NaN,NaN,0.00,177.51,GDDS,NaN,False,2024-10-02 15:06:00,2024-10-02 15:30:32,2024-10-07 07:55:00,20241007,completed,NaN,True,0,False,payments_2024-10.parquet
4191,P00009274,E2E00009274,1bba70e2-4399-450f-83c8-676472f06448,NaN,IN,A006,C0059,11,INBOUND,CN,SG,11727.16,USD,15636.29,NaN,SHA,0.00,11702.57,GDDS,INV-E02-0002757,False,2024-10-08 00:52:00,2024-10-08 00:57:03,2024-10-08 04:49:13,20241008,completed,NaN,True,0,False,payments_2024-10.parquet
4192,P00009649,E2E00009649,NaN,NaN,OUT,A011,C0133,10,LEGACY_FILE,CN,CN,341883.32,CNY,63181.06,NaN,NaN,27.06,341883.32,SUPP,C0133/680335,False,2024-10-10 01:50:00,2024-10-10 05:06:21,2024-10-10 05:28:48,20241010,completed,NaN,True,0,False,payments_2024-10.parquet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106615,P00110638,E2E00110638,NaN,NaN,OUT,A019,C0175,12,H2H_FILE,IN,CN,9755.52,INR,158.45,NaN,NaN,0.00,9755.52,GDDS,NaN,False,2026-09-25 08:00:00,2026-09-25 13:19:44,2026-09-29 04:00:00,20260929,completed,NaN,False,1,False,payments_2026-09.parquet
106616,P00110707,E2E00110707,NaN,NaN,IN,A008,C0230,10,INBOUND,CN,CN,306391.05,CNY,57263.57,NaN,NaN,0.00,306391.05,SCVE,INV-E03-0037165,False,2026-09-28 01:57:00,2026-09-28 01:59:20,2026-09-28 02:16:46,20260928,completed,NaN,True,0,False,payments_2026-09.parquet
106617,P00111287,E2E00111287,7eefd59c-9eb6-4965-a7f4-0654e2bbc373,NaN,OUT,A042,C9090,11,PORTAL,US,VN,152473.56,usd,208157.36,NaN,BEN,0.00,152427.72,SUPP,NaN,False,2026-09-29 15:26:00,2026-09-29 15:52:31,2026-09-30 02:06:00,20260930,completed,NaN,True,0,False,payments_2026-09.parquet
106618,P00111372,E2E00111372,NaN,NaN,OUT,A019,C0123,12,H2H_FILE,IN,IN,10081.66,INR,162.94,NaN,NaN,0.00,NaN,GDDS,NaN,False,2026-09-30 06:27:00,2026-09-30 07:16:48,NaT,<NA>,pending,NaN,True,0,False,payments_2026-09.parquet


### We've found 415 duplicates, so lets remove them and use a new dataframe called "deduped" to continue our analysis

In [10]:
deduped = raw.drop_duplicates(subset="payment_id", keep="first")

In [11]:
deduped.isna().sum().sort_values(ascending=False) # Checks each column for a null value, sums and sorts it

failure_reason      104895
batch_id             98045
fx_rate_applied      94238
charge_bearer        67145
uetr                 67145
remittance_ref       39123
purpose_code          3158
amount_received       1341
value_date_id         1341
settled_ts            1341
is_stp                   0
submitted_ts             0
initiated_ts             0
is_intercompany          0
repair_count             0
missed_cutoff            0
status                   0
payment_id               0
fees                     0
end_to_end_id            0
amount_sgd               0
currency_code            0
amount                   0
receiver_country         0
sender_country           0
channel                  0
type_id                  0
counterparty_id          0
account_id               0
direction                0
_source_file             0
dtype: int64

Still a large number of null values for the 10 columns, so lets tackle it column by column

# Step 3b · Check each null and non-null column against its expected rule

### 1. Every "Purpose Code" is definitely a violation

In [12]:
deduped["purpose_code"].isna().sum()

np.int64(3158)

All 3158 rows are a violation and an error

### 2. batch_id checks

Batch id should only be populated for outbound payments with dim_payment_type is_batch = 1

From dim_payment_type, type_id = 2, 4, 6, 8, 9, 12 are all batch payments

In [13]:
# Check that all non-null values are when
# 1. dim_payment_type is_batch = 1, and
# 2. direction = OUT, and
# 3. type_id =/= 12 (card spend never gets a batch_id)

pt = pd.read_csv(CSV / "dim_payment_type.csv").set_index("type_id")   # lookup table: type_id -> is_batch, is_cross_border... (pt = payment type)

allowed = (deduped["type_id"].map(pt["is_batch"]) == 1) & (deduped["direction"] == "OUT") & (deduped["type_id"] != 12)
non_null_bad = deduped["batch_id"].notna() & ~allowed    # batch_id is populated, but shouldn't be

print(non_null_bad.sum())                                # want 0
deduped[non_null_bad].head()                             # inspect any violating rows

# No check on null values: a null batch_id is allowed on a batch rail (e.g. urgent payments), so null does not imply is_batch = 0

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 3. charge_bearer checks


Charge_bearer is populated when there is a cross border payment, so it should be when dim_payment_type is_cross_border = 1

In [14]:
pt = pd.read_csv(CSV / "dim_payment_type.csv").set_index("type_id")   # lookup table: type_id -> is_batch, is_cross_border... (pt = payment type)

is_xb = deduped["type_id"].map(pt["is_cross_border"]) == 1            # True where the payment's rail is cross-border (xb = cross border)

# Check that all non-null values are when dim_payment_type is_cross_border = 1
populated_but_not_xb = deduped["charge_bearer"].notna() & ~is_xb      # charge_bearer is populated on a domestic payment

print(populated_but_not_xb.sum())                                     # want 0
deduped[populated_but_not_xb].head()                                  # inspect any violating rows

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [15]:
# Check that all null values are when dim_payment_type is_cross_border = 0
missing_on_xb = deduped["charge_bearer"].isna() & is_xb               # charge_bearer is null on a cross-border payment

print(missing_on_xb.sum())                                            # want 0
deduped[missing_on_xb].head()                                         # inspect any violating rows

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 4. uetr checks

uetr is also only populated when there is a cross border payment, so it should be when dim_payment_type is_cross_border = 1

In [16]:
pt = pd.read_csv(CSV / "dim_payment_type.csv").set_index("type_id")

is_xb = deduped["type_id"].map(pt["is_cross_border"]) == 1            # True where the payment's rail is cross-border (xb = cross border)

# Check that all non-null values are when dim_payment_type is_cross_border = 1
populated_but_not_xb_uetr = deduped["uetr"].notna() & ~is_xb          # uetr is populated on a domestic payment

print(populated_but_not_xb_uetr.sum())                                # want 0
deduped[populated_but_not_xb_uetr].head()                             # inspect any violating rows

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [17]:
# Check that all null values are when dim_payment_type is_cross_border = 0
missing_on_xb_uetr = deduped["uetr"].isna() & is_xb                   # uetr is null on a cross-border payment

print(missing_on_xb_uetr.sum())                                       # want 0
deduped[missing_on_xb_uetr].head()                                    # inspect any violating rows

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 5. fx_rate_applied checks

For this dataset, fx_rate_applied is only populated when the account's currency differs from the payment currency, and the status is not rejected or pending.

Result: 516 rows are flagged as null where a rate should exist. All 516 have a dirty currency_code (e.g. `US$`, `usd`, `sgd`, `SGP`, and `'EUR '` with a trailing space), so the code looks different from the account's currency. This is the currency-code defect, see 5b. The rule itself holds.

In [18]:
# check that all null values are when 
# 1. currency_code = [dim_account currency_code, looked up using account_id], or
# 2. status = rejected or pending


# check that all non-null values are when
# 1. currency_code =/= [dim_account currency_code, looked up using account_id], and
# 2. status = completed | delayed | returned

acct = pd.read_csv(CSV / "dim_account.csv").set_index("account_id")["currency_code"]   # lookup table: account_id -> account's currency
acct_ccy = deduped["account_id"].map(acct)                                             # the account's currency for every payment row

needs_fx = (deduped["currency_code"] != acct_ccy) & deduped["status"].isin(["completed", "delayed", "returned"])   # a rate should exist

null_bad     = deduped["fx_rate_applied"].isna() & needs_fx      # rate missing where it should exist
non_null_bad = deduped["fx_rate_applied"].notna() & ~needs_fx    # rate present where it shouldn't be

print(null_bad.sum(), non_null_bad.sum())                        # want 0 and 0
deduped[null_bad]["currency_code"].value_counts()                # inspect what the violating rows look like

516 0


currency_code
usd     83
RMB     68
EUR     66
eur     65
US$     64
cny     61
sgd     33
SGP     30
inr     19
INR     10
GBP     10
gbp      7
Name: count, dtype: int64

### 6. remittance_ref checks checks

There's no strict rule for this since it is an additional text field "comment". I'll defer it for later if necessary

### 7. failure_reason checks

Every populated failure_reason should have a "rejected" or "returned" status, and every rejected or returned payment should have one

In [19]:
# Check that all null values are when "status" = "rejected" | "returned"
no_failure = deduped["status"].isin(["completed", "pending", "delayed"])    # statuses that should have NO failure_reason
null_bad_failure = deduped["failure_reason"].isna() & ~no_failure           # failure_reason is null on a rejected/returned payment

print(null_bad_failure.sum())                                               # want 0
deduped[null_bad_failure].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [20]:
# Check that all non-null values are when "status" = "rejected" | "returned"
non_null_bad_failure = deduped["failure_reason"].notna() & no_failure       # failure_reason is populated on a payment that didn't fail

print(non_null_bad_failure.sum())                                           # want 0
deduped[non_null_bad_failure].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 8. amount_received checks

Amount_received should be null when the payment status is rejected or pending, and populated when it is completed, delayed or returned

Amount_received should also always be less than or equal to the absolute amount (some amounts are negative, see 5d). Receiving more than what was sent is a defect

In [21]:
# Check that all null "Amount_received" values are when payment status = "rejected" | "pending"
amt_filter = deduped["status"].isin(["rejected","pending"]) # These are the only values for status when amount_received should be null

null_bad_amount_received = deduped["amount_received"].isna() & ~amt_filter

print(null_bad_amount_received.sum())
deduped[null_bad_amount_received].head()


0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [22]:
# Check that all non-null "Amount_received" values are when payment status = "completed" | "delayed" | "returned"
non_null_bad_amount_received = deduped["amount_received"].notna() & amt_filter    # amount_received is populated on a rejected/pending payment

print(non_null_bad_amount_received.sum())                                         # want 0
deduped[non_null_bad_amount_received].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [23]:
# Additionally, check for each non-null "Amount_received" row that "Amount_received <= abs("Amount")
amount_errors = deduped["amount_received"] <= deduped["amount"].abs()

non_null_bad_amount_errors = deduped["amount_received"].notna() & ~amount_errors    # .notna() so rejected/pending rows are not flagged
print(non_null_bad_amount_errors.sum())                                              # want 0
deduped[non_null_bad_amount_errors].head()

# Without .abs(), the print returned 291, which is due to the "amount"s being negative (the 5d defect).

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 9. value_date_id checks

value_date_id should only be populated when a payment status is completed, delayed, or returned. It is null when the payment is rejected or still pending

In [24]:
# Check that all null "value_date_id" values are when payment status = "rejected" | "pending"
value_date_id_filters = deduped["status"].isin(["pending","rejected"])

null_bad_date_id = deduped["value_date_id"].isna() & ~value_date_id_filters
print(null_bad_date_id.sum())
deduped[null_bad_date_id].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [25]:
# Check that all non-null "value_date_id" values are when payment status = "completed" | "delayed" | "returned"
non_null_bad_date_id = deduped["value_date_id"].notna() & value_date_id_filters
print(non_null_bad_date_id.sum())
deduped[non_null_bad_date_id].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### 10. settled_ts checks

settled_ts should only be populated when a payment status is completed, delayed, or returned. It is null when the payment is rejected or still pending

In [26]:
# Check that all null "settled_ts" values are when payment status = "rejected" | "pending"
settled_ts_filters = deduped["status"].isin(["pending","rejected"])

null_bad_settled_ts = deduped["settled_ts"].isna() & ~settled_ts_filters
print(null_bad_settled_ts.sum())                                           # want 0
deduped[null_bad_settled_ts].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


In [27]:
# Check that all non-null "settled_ts" values are when payment status = "completed" | "delayed" | "returned"
non_null_bad_settled_ts = deduped["settled_ts"].notna() & settled_ts_filters
print(non_null_bad_settled_ts.sum())                                       # want 0
deduped[non_null_bad_settled_ts].head()

0


,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file


### Summary of the null-rule checks (on `deduped`, 106,206 rows)

| Field | Rule that was tested | Violations | Verdict |
|---|---|---|---|
| `purpose_code` | always populated | 3,158 null | Defect (5f) |
| `batch_id` | only populated for OUT, is_batch = 1, not card (type 12). Null is allowed on batch rails, so only this direction is checked | 0 | Nulls are expected |
| `charge_bearer` | populated if and only if is_cross_border = 1 | 0 / 0 | Nulls are expected |
| `uetr` | populated if and only if is_cross_border = 1 | 0 / 0 | Nulls are expected |
| `fx_rate_applied` | populated if and only if currency_code =/= account currency and status is completed, delayed or returned | 516 null where a rate should exist, 0 the other way | Rule holds. The 516 are dirty currency codes (5b) |
| `remittance_ref` | no strict rule | not checked | Deferred. Matters for reconciliation (analysis 8) |
| `failure_reason` | populated if and only if status is rejected or returned | 0 / 0 | Nulls are expected |
| `amount_received` | null if and only if status is rejected or pending. Always <= abs(amount) | 0 / 0, and 0 for the amount check | Nulls are expected. 291 rows only break `amount_received <= amount` because amount is negative (5d) |
| `value_date_id` | null if and only if status is rejected or pending | 0 / 0 | Nulls are expected |
| `settled_ts` | null if and only if status is rejected or pending | 0 / 0 | Nulls are expected |

Only `purpose_code` is a real null defect. The other nulls are explained by the business rules, and the one non-zero count (`fx_rate_applied`) comes from the currency-code defect, which is looked at in 5b.

## Step 4 · Raw vs clean

The clean database holds the simulator's clean version of the same payments. Comparing row counts is the fastest sanity check a data engineer does.

In [28]:
clean_count = len(pd.read_csv(CSV / "fact_payment.csv"))
print(f"raw      : {len(raw):>9,}")
print(f"clean    : {clean_count:>9,}")
print(f"extra    : {len(raw) - clean_count:>9,}   <- where do these come from? (Step 5a)")

raw      :   106,621
clean    :   106,206
extra    :       415   <- where do these come from? (Step 5a)


## Step 5 · Finding the defects

Each finding below ends with a notes cell that answers:
1. **What** is wrong, and how many rows?
2. **Can it be fixed** with certainty, or only guessed?
3. **What check** (one line of pandas) would catch it automatically?

In [29]:
VALID_CCY = set(pd.read_csv(CSV / "dim_currency.csv")["currency_code"])
VALID_CTRY = set(pd.read_csv(CSV / "dim_country.csv")["country_code"])
print(VALID_CCY)
print(sorted(VALID_CTRY))

{'EUR', 'SGD', 'INR', 'CNY', 'GBP', 'USD'}
['AE', 'AU', 'CN', 'DE', 'GB', 'ID', 'IN', 'JP', 'MY', 'NL', 'SG', 'US', 'VN', 'XA', 'XB']


### 5a · Duplicates
`payment_id` should be unique: one row per payment.

Two kinds of "duplicate" look the same in the data but are different in real life:
- a **re-sent file**: the same payment delivered twice (a data problem, the payment only happened once)
- an **actual duplicate payment**: someone really paid twice, so there are two different `payment_id`s (a business problem, the money really moved twice)

Checking `payment_id` only finds the first kind. The second kind belongs to analysis 9 (anomaly detection).

In [30]:
raw["payment_id"].duplicated().sum()

np.int64(415)

In [31]:
raw.duplicated().sum() # This means that the entire row are exact duplicates, and it is not just a duplicated payment_id

np.int64(415)

In [32]:
dups = raw[raw.duplicated(keep=False)].sort_values("payment_id")
dups.head(10)   # 10 rows = 5 pairs

,payment_id,end_to_end_id,uetr,batch_id,direction,account_id,counterparty_id,type_id,channel,sender_country,receiver_country,amount,currency_code,amount_sgd,fx_rate_applied,charge_bearer,fees,amount_received,purpose_code,remittance_ref,is_intercompany,initiated_ts,submitted_ts,settled_ts,value_date_id,status,failure_reason,is_stp,repair_count,missed_cutoff,_source_file
12,P00008598,E2E00008598,29d85190-de9c-4846-a7fd-04f7804d28dd,NaN,IN,A028,C0108,11,INBOUND,ID,DE,7059.55,US$,9489.49,NaN,BEN,0.00,7030.63,GDDS,NaN,False,2024-10-01 03:02:00,2024-10-01 03:06:28,2024-10-01 03:44:58,20241001,completed,NaN,True,0,False,payments_2024-10.parquet
4188,P00008598,E2E00008598,29d85190-de9c-4846-a7fd-04f7804d28dd,NaN,IN,A028,C0108,11,INBOUND,ID,DE,7059.55,US$,9489.49,NaN,BEN,0.00,7030.63,GDDS,NaN,False,2024-10-01 03:02:00,2024-10-01 03:06:28,2024-10-01 03:44:58,20241001,completed,NaN,True,0,False,payments_2024-10.parquet
137,P00008723,E2E00008723,NaN,NaN,OUT,A020,C0140,12,H2H_FILE,DE,DE,233.01,EUR,345.76,NaN,NaN,0.00,233.01,GDDS,NaN,False,2024-10-01 10:43:00,2024-10-01 16:07:49,2024-10-04 07:15:00,20241004,completed,NaN,False,1,False,payments_2024-10.parquet
4189,P00008723,E2E00008723,NaN,NaN,OUT,A020,C0140,12,H2H_FILE,DE,DE,233.01,EUR,345.76,NaN,NaN,0.00,233.01,GDDS,NaN,False,2024-10-01 10:43:00,2024-10-01 16:07:49,2024-10-04 07:15:00,20241004,completed,NaN,False,1,False,payments_2024-10.parquet
243,P00008829,E2E00008829,NaN,NaN,OUT,A020,C0134,12,H2H_FILE,DE,DE,177.51,EUR,264.12,NaN,NaN,0.00,177.51,GDDS,NaN,False,2024-10-02 15:06:00,2024-10-02 15:30:32,2024-10-07 07:55:00,20241007,completed,NaN,True,0,False,payments_2024-10.parquet
4190,P00008829,E2E00008829,NaN,NaN,OUT,A020,C0134,12,H2H_FILE,DE,DE,177.51,EUR,264.12,NaN,NaN,0.00,177.51,GDDS,NaN,False,2024-10-02 15:06:00,2024-10-02 15:30:32,2024-10-07 07:55:00,20241007,completed,NaN,True,0,False,payments_2024-10.parquet
688,P00009274,E2E00009274,1bba70e2-4399-450f-83c8-676472f06448,NaN,IN,A006,C0059,11,INBOUND,CN,SG,11727.16,USD,15636.29,NaN,SHA,0.00,11702.57,GDDS,INV-E02-0002757,False,2024-10-08 00:52:00,2024-10-08 00:57:03,2024-10-08 04:49:13,20241008,completed,NaN,True,0,False,payments_2024-10.parquet
4191,P00009274,E2E00009274,1bba70e2-4399-450f-83c8-676472f06448,NaN,IN,A006,C0059,11,INBOUND,CN,SG,11727.16,USD,15636.29,NaN,SHA,0.00,11702.57,GDDS,INV-E02-0002757,False,2024-10-08 00:52:00,2024-10-08 00:57:03,2024-10-08 04:49:13,20241008,completed,NaN,True,0,False,payments_2024-10.parquet
1063,P00009649,E2E00009649,NaN,NaN,OUT,A011,C0133,10,LEGACY_FILE,CN,CN,341883.32,CNY,63181.06,NaN,NaN,27.06,341883.32,SUPP,C0133/680335,False,2024-10-10 01:50:00,2024-10-10 05:06:21,2024-10-10 05:28:48,20241010,completed,NaN,True,0,False,payments_2024-10.parquet
4192,P00009649,E2E00009649,NaN,NaN,OUT,A011,C0133,10,LEGACY_FILE,CN,CN,341883.32,CNY,63181.06,NaN,NaN,27.06,341883.32,SUPP,C0133/680335,False,2024-10-10 01:50:00,2024-10-10 05:06:21,2024-10-10 05:28:48,20241010,completed,NaN,True,0,False,payments_2024-10.parquet


In [33]:
# Pick one repeated payment and show both copies side by side (columns = the two copies)
one_id = dups["payment_id"].iloc[0]
raw[raw["payment_id"] == one_id].T

,12,4188
payment_id,P00008598,P00008598
end_to_end_id,E2E00008598,E2E00008598
uetr,29d85190-de9c-4846-a7fd-04f7804d28dd,29d85190-de9c-4846-a7fd-04f7804d28dd
batch_id,NaN,NaN
direction,IN,IN
account_id,A028,A028
counterparty_id,C0108,C0108
type_id,11,11
channel,INBOUND,INBOUND
sender_country,ID,ID


In [34]:
# For every repeated payment_id, count how many columns differ between the copies (ignoring _source_file),
# and how many repeated payment_ids had both copies inside the same monthly file
check_cols = [c for c in raw.columns if c != "_source_file"]
columns_that_differ = dups.groupby("payment_id")[check_cols].nunique(dropna=False).gt(1).sum().sum()
same_file = (dups.groupby("payment_id")["_source_file"].nunique() == 1).sum()

print("columns that differ between copies:", columns_that_differ)
print("repeated payment_ids:", dups["payment_id"].nunique(), "| both copies in the same file:", same_file)

columns that differ between copies: 0
repeated payment_ids: 415 | both copies in the same file: 415


**Notes 5a:**

1. **What:** 415 rows repeat a `payment_id` that already appeared. Every repeated id appears exactly twice, and the two copies are identical in every column. All 415 pairs sit inside the same monthly file. This is a re-sent row, not two real payments (two real payments would have two different `payment_id`s).
2. **Fixable with certainty:** yes. Keep the first row per `payment_id` and drop the copy, because the copy carries no extra information.
3. **Check:** `raw["payment_id"].duplicated().sum()`

Dropping on `payment_id` is safe. Dropping on something like (counterparty, amount, date) would be dangerous, because that would also delete a *genuine* duplicate payment, which did post to the ledger.

### 5b · Currency codes
The valid codes are `VALID_CCY` (from `dim_currency`). Anything else is a defect. From here on I use `deduped`, so the duplicate rows from 5a are not counted twice.

In [35]:
deduped["currency_code"].value_counts()

currency_code
USD     30543
EUR     26859
CNY     25750
SGD     11177
GBP      6198
INR      5094
usd        84
eur        78
RMB        77
EUR        75
cny        74
US$        65
sgd        42
SGP        33
inr        20
GBP        17
INR        10
gbp        10
Name: count, dtype: int64

In [36]:
bad_currency_codes_rows = deduped[~deduped["currency_code"].isin(VALID_CCY)]
bad_currency_codes_rows["currency_code"].value_counts()

currency_code
usd     84
eur     78
RMB     77
EUR     75
cny     74
US$     65
sgd     42
SGP     33
inr     20
GBP     17
INR     10
gbp     10
Name: count, dtype: int64

In [37]:
# Map the odd spellings to the real codes. .strip().upper() already handles "EUR " and "eur", "gbp", "inr" etc.
# so the map only needs the spellings that are not just a different case or a trailing space.
CCY_MAP = {"US$": "USD", "SGP": "SGD", "RMB": "CNY"}
fixed_currencies = deduped["currency_code"].str.strip().replace(CCY_MAP).str.upper()

fixed_currencies.isin(VALID_CCY).value_counts()

currency_code
True    106206
Name: count, dtype: int64

Same check as 5c: is the map right, not just valid? I compare the fixed code with the true currency in the clean database (only allowed in the notebook to learn; the pipeline can't use the clean database).

In [38]:
wh = pd.read_sql("SELECT payment_id, currency_code AS true_ccy FROM fact_payment", con)
bad_vs_true = deduped[~deduped["currency_code"].isin(VALID_CCY)].merge(wh, on="payment_id")

fixed = bad_vs_true["currency_code"].str.strip().replace(CCY_MAP).str.upper()
(fixed == bad_vs_true["true_ccy"]).sum(), len(bad_vs_true)

(np.int64(585), 585)

**Notes 5b:**

1. **What:** 585 rows (after removing duplicates) have a currency code that is not in `VALID_CCY`. They are misspellings of the real code: lower case (`usd`, `eur`), a trailing space (`EUR `, `GBP `), or a different symbol (`US$`, `SGP`, `RMB`). 516 of them are the rows with a missing `fx_rate_applied` from step 3b: the dirty code made the payment look like it was in the account's own currency, so no rate was applied.
2. **Fixable with certainty:** yes. Each bad spelling belongs to exactly one currency, and the fixed code is right on all 585 rows.
3. **Check:** `~deduped["currency_code"].isin(VALID_CCY)`

### 5c · Country names
The valid codes are `VALID_CTRY` (from `dim_country`). I look at both `sender_country` and `receiver_country`, list every value that is not a valid code, and write down the code each should map to. That dictionary becomes `COUNTRY_MAP` in `clean.py`.

In [39]:
print(deduped["sender_country"].value_counts().to_string())

sender_country
CN                24988
DE                20047
US                17908
SG                15677
IN                 6474
GB                 6448
NL                 6217
VN                 2539
MY                 2264
AE                  848
XB                  694
ID                  633
JP                  156
XA                  150
AU                  130
PRC                  96
China                87
cn                   86
Singapore            75
Germany              73
de                   72
Deutschland          66
USA                  63
U.S.                 57
us                   56
sg                   46
S'pore               41
India                36
Netherlands          33
United Kingdom       30
in                   30
nl                   25
Holland              23
UK                   21
gb                   17


In [40]:
for col in ["sender_country", "receiver_country"]:
    print(col, sorted(set(deduped[col].dropna()) - VALID_CTRY))

sender_country ['China', 'Deutschland', 'Germany', 'Holland', 'India', 'Netherlands', 'PRC', "S'pore", 'Singapore', 'U.S.', 'UK', 'USA', 'United Kingdom', 'cn', 'de', 'gb', 'in', 'nl', 'sg', 'us']
receiver_country []


In [41]:
bad_ctry = deduped.loc[~deduped["sender_country"].isin(VALID_CTRY), "sender_country"]
print(bad_ctry.value_counts().to_string())

sender_country
PRC               96
China             87
cn                86
Singapore         75
Germany           73
de                72
Deutschland       66
USA               63
U.S.              57
us                56
sg                46
S'pore            41
India             36
Netherlands       33
United Kingdom    30
in                30
nl                25
Holland           23
UK                21
gb                17


In [42]:
COUNTRY_MAP = {
    "China": "CN", "PRC": "CN", "cn": "CN",
    "Germany": "DE", "Deutschland": "DE", "de": "DE",
    "Netherlands": "NL", "Holland": "NL", "nl": "NL",
    "India": "IN", "in": "IN",
    "Singapore": "SG", "S'pore": "SG", "sg": "SG",
    "United Kingdom": "GB", "UK": "GB", "gb": "GB",
    "USA": "US", "U.S.": "US", "us": "US",
}

Same lesson as 5b: before trusting the map, check it against the clean database. The map is only safe if every messy value always means one country.

In [43]:
wh_ctry = pd.read_sql("SELECT payment_id, sender_country AS true_ctry FROM fact_payment", con)
bad_ctry_rows = deduped[~deduped["sender_country"].isin(VALID_CTRY)].merge(wh_ctry, on="payment_id")

# rows = the messy value in the raw file, columns = the country it really was
pd.crosstab(bad_ctry_rows["sender_country"], bad_ctry_rows["true_ctry"])

true_ctry,CN,DE,GB,IN,NL,SG,US
sender_country,,,,,,,
China,87,0,0,0,0,0,0
Deutschland,0,66,0,0,0,0,0
Germany,0,73,0,0,0,0,0
Holland,0,0,0,0,23,0,0
India,0,0,0,36,0,0,0
Netherlands,0,0,0,0,33,0,0
PRC,96,0,0,0,0,0,0
S'pore,0,0,0,0,0,41,0
Singapore,0,0,0,0,0,75,0


In [44]:
# How often does COUNTRY_MAP give the right answer?
(bad_ctry_rows["sender_country"].map(COUNTRY_MAP) == bad_ctry_rows["true_ctry"]).sum(), len(bad_ctry_rows)

(np.int64(1033), 1033)

In [45]:
fixed_country = deduped["sender_country"].str.strip().replace(COUNTRY_MAP).str.upper()

fixed_country.isin(VALID_CTRY).value_counts()

sender_country
True    106206
Name: count, dtype: int64

**Notes 5c:**

1. **What:** 1,033 rows have a `sender_country` written as a name or in the wrong case ("China", "PRC", "cn", "Deutschland", "UK", "U.S." ...), spread over 20 messy values. `receiver_country` has no bad values.
2. **Fixable with certainty:** yes. In the crosstab every messy value lands in exactly one column, so no value can mean two countries, and `COUNTRY_MAP` is right for all 1,033 rows. (Unlike the currency codes, the messy text really does say what the country is.)
3. **Check:** `~deduped["sender_country"].isin(VALID_CTRY)`

### 5d · Negative amounts
`amount` should always be positive, because the `direction` column (OUT or IN) already says which way the money moves. I check how many rows are negative, whether `amount_sgd` and `amount_received` are negative too, and what the minus sign adds on top of `direction`.

In [46]:
negative_amounts = deduped[deduped["amount"]<0]
len(negative_amounts)

293

In [47]:
(negative_amounts["amount_sgd"]< 0 ).value_counts()

amount_sgd
False    293
Name: count, dtype: int64

In [48]:
(negative_amounts["amount_received"] < 0).value_counts()

amount_received
False    293
Name: count, dtype: int64

In [49]:
negative_amounts["direction"].value_counts()

direction
OUT    194
IN      99
Name: count, dtype: int64

In [50]:
# Are the negative rows still consistent once the sign is removed? (amount_received <= abs(amount))
(negative_amounts["amount_received"] <= negative_amounts["amount"].abs()).sum(), negative_amounts["amount_received"].notna().sum()

(np.int64(291), np.int64(291))

In [51]:
# Does abs(amount) match the amount in the clean database?
wh_amt = pd.read_sql("SELECT payment_id, amount AS true_amount FROM fact_payment", con)
neg_vs_true = negative_amounts.merge(wh_amt, on="payment_id")
(neg_vs_true["amount"].abs() == neg_vs_true["true_amount"]).sum(), len(neg_vs_true)

(np.int64(293), 293)

**Notes 5d:**

1. **What:** 293 rows have a negative `amount` (194 OUT, 99 IN, so the sign is not tied to direction). `amount_sgd` is positive on all 293 and `amount_received` is positive on all 291 that have one. Only `amount` carries the minus sign. This is also the reason 291 rows broke `amount_received <= abs(amount)` in step 3b.
2. **Fixable with certainty:** yes. The sign adds no information because `direction` already says OUT or IN. `abs(amount)` matches the clean database on all 293 rows, and the `amount_received` check passes on all 291 afterwards. It is an export bug, not a refund or a reversal.
3. **Check:** `deduped["amount"] < 0`

### 5e · Timestamps in the wrong order
A payment must be *initiated* → *submitted* → *settled*, in that order. I count the rows that break this, then look at the event feed (`payment_events`) to see which timestamp I can trust.

In [52]:
settled_before_initiated = deduped[deduped["settled_ts"] < deduped["initiated_ts"]]
submitted_before_initiated = deduped[deduped["submitted_ts"] < deduped["initiated_ts"]]

len(settled_before_initiated), len(submitted_before_initiated)

(536, 0)

In [53]:
# Load the event feed the same way as the payments (one parquet file per month)
event_files = sorted((RAW / "payment_events").glob("*.parquet"))
events = pd.concat([pd.read_parquet(f) for f in event_files], ignore_index=True)

# Take one bad payment: the row's timestamps, then its events in time order
bad_id = settled_before_initiated["payment_id"].iloc[0]
print(deduped.loc[deduped["payment_id"] == bad_id, ["initiated_ts", "submitted_ts", "settled_ts"]].T)
events[events["payment_id"] == bad_id].sort_values("event_ts")

                             121
initiated_ts 2024-10-01 09:11:00
submitted_ts 2024-10-01 09:25:01
settled_ts   2024-10-01 07:33:40


,event_id,payment_id,event_ts,status,bank_id,hop_seq,reason_code
450,33989,P00008707,2024-10-01 09:11:00,CREATED,NaN,<NA>,NaN
1970,35682,P00008707,2024-10-04 08:40:00,SETTLED,NaN,<NA>,NaN


In [54]:
# One SETTLED event per payment (checked: there are never two)
settled_events = events[events["status"] == "SETTLED"][["payment_id", "event_ts"]]

# 1. On the healthy rows, does settled_ts always equal the SETTLED event time?
healthy = deduped[deduped["settled_ts"].notna() & ~deduped["payment_id"].isin(settled_before_initiated["payment_id"])]
healthy = healthy.merge(settled_events, on="payment_id")
print("healthy rows where settled_ts == SETTLED event:", (healthy["settled_ts"] == healthy["event_ts"]).mean())

# 2. For the bad rows: is there a SETTLED event, and is it after initiated_ts?
bad_rows = settled_before_initiated.merge(settled_events, on="payment_id", how="left")
print("bad rows with no SETTLED event:", bad_rows["event_ts"].isna().sum())
print("bad rows where the event is after initiated_ts:", (bad_rows["event_ts"] >= bad_rows["initiated_ts"]).sum(), "of", len(bad_rows))

# 3. Does the event time equal the settled_ts in the clean database?
wh_ts = pd.read_sql("SELECT payment_id, settled_ts AS true_settled_ts FROM fact_payment", con)
bad_vs_true_ts = bad_rows.merge(wh_ts, on="payment_id")
print("event time == clean database settled_ts:", (pd.to_datetime(bad_vs_true_ts["true_settled_ts"]) == bad_vs_true_ts["event_ts"]).sum())

healthy rows where settled_ts == SETTLED event: 1.0
bad rows with no SETTLED event: 0
bad rows where the event is after initiated_ts: 536 of 536
event time == clean database settled_ts: 536


**Notes 5e:**

1. **What:** 536 rows have `settled_ts` *before* `initiated_ts` (between 1 hour and 1 day earlier). `submitted_ts` is never before `initiated_ts`, so the broken field is `settled_ts` only. These are the same 536 rows where `settled_ts` is also before `submitted_ts`.
2. **Fixable with certainty:** yes. The event log is append-only (events are only ever added, never edited), so I trust it over the row. On every healthy row `settled_ts` equals the time of the `SETTLED` event exactly, all 536 bad rows have a `SETTLED` event, and in every case it is after `initiated_ts`. It also matches the clean database. Fix: take `settled_ts` from the `SETTLED` event.
3. **Check:** `deduped["settled_ts"] < deduped["initiated_ts"]`

### 5f · Missing purpose codes
`purpose_code` should always be populated (step 3b). I count the nulls, check whether anything reliably says what the code should have been, and decide whether the row should be blocked or loaded with a flag.

In [55]:
purpose_nulls = deduped[deduped["purpose_code"].isna()]
print("null purpose codes:", len(purpose_nulls))
print(purpose_nulls["status"].value_counts().to_dict())

null purpose codes: 3158
{'completed': 2810, 'delayed': 305, 'rejected': 29, 'pending': 8, 'returned': 6}


In [56]:
# Could the counterparty tell me the purpose? Count the distinct purpose codes each counterparty uses.
deduped.groupby("counterparty_id")["purpose_code"].nunique().value_counts().sort_index()

purpose_code
0     14
1    386
2    199
3      6
Name: count, dtype: int64

In [57]:
# Could the most common code be used as a guess? How often is it the most common one?
deduped["purpose_code"].value_counts(normalize=True)

purpose_code
GDDS    0.559894
SUPP    0.305508
SCVE    0.094703
INTC    0.033528
SALA    0.003853
TAXS    0.002513
Name: proportion, dtype: float64

In [58]:
# Try the best guess I can build from the raw data: each counterparty's most common purpose code.
# Then compare it with the real purpose code in the clean database (only possible in the notebook).
usual_purpose = deduped.dropna(subset=["purpose_code"]).groupby("counterparty_id")["purpose_code"].agg(lambda s: s.mode().iloc[0])
wh_purpose = pd.read_sql("SELECT payment_id, purpose_code AS true_purpose FROM fact_payment", con)

guess = purpose_nulls.merge(wh_purpose, on="payment_id")
guess["guessed"] = guess["counterparty_id"].map(usual_purpose)
print(f"counterparty guess is right on {(guess['guessed'] == guess['true_purpose']).mean():.0%} of the {len(guess)} rows")

counterparty guess is right on 61% of the 3158 rows


**Notes 5f:**

1. **What:** 3,158 rows have no `purpose_code` (all statuses, mostly completed). That is about 3% of the table.
2. **Fixable with certainty:** no. About a third of counterparties use 2 or 3 different purpose codes, so the counterparty alone doesn't decide it. The most common code (`GDDS`) is only ~56% of the populated rows, and the best guess I could build from the raw data (each counterparty's usual code) is right on only ~61% of the null rows, so about 4 in 10 would be wrong. Any fill-in would be a guess dressed up as data.
3. **Check:** `deduped["purpose_code"].isna()`

Decision: **don't fix and don't block.** The payment is real money and still counts in cash totals, so blocking it would hide it from treasury. The row is loaded with the null kept and a flag on it, so the missing purpose can be chased with the source system.

### 5g · Anything else?
The six hunts above are the ones I expected. A last sweep over the other columns for anything that doesn't look right: statuses, decimals, `amount_received`, the date range, foreign keys to the dimensions, and repeated `end_to_end_id`.

In [59]:
print(deduped["status"].value_counts().to_dict())
print("amounts with more than 2 decimals:", (deduped["amount"].round(2) != deduped["amount"]).sum())
print("amount_received > abs(amount):", (deduped["amount_received"] > deduped["amount"].abs()).sum())
print("date range:", deduped["initiated_ts"].min(), "to", deduped["initiated_ts"].max())
print("zero amounts:", (deduped["amount"] == 0).sum(), "| negative fees:", (deduped["fees"] < 0).sum())

{'completed': 94635, 'delayed': 10047, 'rejected': 1128, 'pending': 213, 'returned': 183}
amounts with more than 2 decimals: 0
amount_received > abs(amount): 0
date range: 2024-10-01 01:05:00 to 2026-10-01 01:58:00
zero amounts: 0 | negative fees: 0


In [60]:
# Do the foreign keys exist in the dimensions?
accounts = pd.read_csv(CSV / "dim_account.csv")["account_id"]
counterparties = pd.read_csv(CSV / "dim_counterparty.csv")["counterparty_id"]
print("unknown account_id     :", (~deduped["account_id"].isin(accounts)).sum())
print("unknown counterparty_id:", (~deduped["counterparty_id"].isin(counterparties)).sum())

# Repeated end_to_end_id: expected for intercompany payments (two legs, OUT and IN)
repeated_e2e = deduped[deduped["end_to_end_id"].duplicated(keep=False)]
print("repeated end_to_end_id rows:", len(repeated_e2e), "| of which not intercompany:", (repeated_e2e["is_intercompany"] == 0).sum())

unknown account_id     : 0
unknown counterparty_id: 0
repeated end_to_end_id rows: 3544 | of which not intercompany: 0


**Notes 5g:** nothing new found.

- `status` only has the five expected values, there are no zero amounts, no negative fees and no amount with more than 2 decimals.
- `amount_received` is never above `abs(amount)`.
- The dates run from 2024-10-01 to 2026-10-01 and all `account_id` and `counterparty_id` exist in the dimensions.
- `end_to_end_id` repeats on 3,544 rows (1,772 pairs), and all of them are intercompany payments. That is expected: an intercompany payment has two legs (OUT and IN) that share the same id. It is only a problem if a group total forgets to filter `is_intercompany`.

## Step 6 · Defect table

Everything from Step 5 in one place. It becomes the decision table at the top of `src/treasury/pipeline/clean.py` and a section of the write-up. Row counts are on `deduped`, except duplicates (counted on `raw`).

| Defect | Rows found | Check (one line) | Fix or quarantine? | Evidence for the fix |
|---|---|---|---|---|
| Duplicates | 415 | `raw["payment_id"].duplicated()` | Fix: drop the copy, keep the first row | All 415 pairs are identical in every column and sit in the same monthly file, so the copy adds nothing |
| Currency codes | 585 | `~deduped["currency_code"].isin(VALID_CCY)` | Fix: `.str.strip().str.upper()` plus `CCY_MAP` | Every bad spelling belongs to one currency, and the fixed code is right on all 585 rows |
| Country names | 1,033 | `~deduped["sender_country"].isin(VALID_CTRY)` | Fix: `COUNTRY_MAP` | Every messy value maps to one country, and the map is right on all 1,033 rows |
| Negative amounts | 293 | `deduped["amount"] < 0` | Fix: `abs(amount)` | `direction` already says OUT or IN. `amount_sgd` and `amount_received` are already positive. `abs(amount)` matches the clean database on all 293 rows |
| Time order | 536 | `deduped["settled_ts"] < deduped["initiated_ts"]` | Fix: `settled_ts` from the `SETTLED` event | The event log is append-only. It equals `settled_ts` on every healthy row and is after `initiated_ts` on all 536 bad rows |
| Null purpose code | 3,158 | `deduped["purpose_code"].isna()` | Don't fix: load with a flag | No reliable way to know the value (a counterparty-based guess is wrong ~4 in 10 times), and blocking would hide real money |

## Step 7 · Open the answer key

Only now. `dq_defects.parquet` lists every planted defect. I compare it with my Step 6 table to see whether I missed a type or over-counted one.

In [61]:
labels = pd.read_parquet(ANSWER_KEY / "dq_defects.parquet")
labels["defect_type"].value_counts()

defect_type
null_purpose_code            3158
inconsistent_country_name    1033
bad_currency_code             585
time_order_violation          536
resent_file_duplicate         415
negative_amount               293
Name: count, dtype: int64

In [62]:
# My checks from Step 6, each as a set of payment_ids
mine = {
    "resent_file_duplicate":     set(raw.loc[raw["payment_id"].duplicated(), "payment_id"]),
    "bad_currency_code":         set(deduped.loc[~deduped["currency_code"].isin(VALID_CCY), "payment_id"]),
    "inconsistent_country_name": set(deduped.loc[~deduped["sender_country"].isin(VALID_CTRY), "payment_id"]),
    "negative_amount":           set(deduped.loc[deduped["amount"] < 0, "payment_id"]),
    "time_order_violation":      set(deduped.loc[deduped["settled_ts"] < deduped["initiated_ts"], "payment_id"]),
    "null_purpose_code":         set(deduped.loc[deduped["purpose_code"].isna(), "payment_id"]),
}

rows = []
for defect, found in mine.items():
    truth = set(labels.loc[labels["defect_type"] == defect, "payment_id"])
    hit = len(found & truth)
    rows.append({"defect": defect, "mine": len(found), "answer key": len(truth),
                 "precision": hit / len(found), "recall": hit / len(truth)})
pd.DataFrame(rows)

,defect,mine,answer key,precision,recall
0,resent_file_duplicate,415,415,1.0,1.0
1,bad_currency_code,585,585,1.0,1.0
2,inconsistent_country_name,1033,1033,1.0,1.0
3,negative_amount,293,293,1.0,1.0
4,time_order_violation,536,536,1.0,1.0
5,null_purpose_code,3158,3158,1.0,1.0


**Notes on Step 7:**

My counts match the answer key on all six defect types, and for each one the set of payment_ids is the same (precision and recall are both 1.0). I didn't miss a type and didn't over-count one. The answer key also confirms that nothing else was planted, so the "anything else?" sweep in 5g correctly found nothing new.

Every defect except the null purpose codes can be fixed with certainty. The duplicates are dropped, the negative amounts, country names and currency codes are fixed with a rule or a map, and the time order is taken from the event log. The null purpose codes can't be fixed, so they are kept and flagged.

## Next step

The cleaning pipeline is skipped on purpose. From here the project uses the clean database, `data_small/clean/treasury.sqlite`, and the SQL analyses in `sql/analyses/`.